# 07 — Case-Based Map Design Challenge
**Part 7 of 7** | GeoMetric Project

## Three real-world map design problems:
- **Scenario A:** Public Health — disease burden, clustering, spread
- **Scenario B:** Urban Services — airport accessibility gaps  
- **Scenario C:** Climate Risk — bivariate hazard × exposure map

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import warnings; warnings.filterwarnings("ignore")
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.interpolate import RBFInterpolator
from scripts.utils.config import PATHS, STYLE
from scripts.utils.map_utils import reproject_gdf, save_figure, add_map_annotations, scale_symbols, points_to_gdf

In [ ]:
world    = gpd.read_file(PATHS["processed"] / "master_world.gpkg")
airports = pd.read_csv(PATHS["processed"] / "airports_clean.csv")
temps    = pd.read_csv(PATHS["processed"] / "temperature_stations.csv")
print("All data loaded ✅")

In [ ]:
# ── SCENARIO A: Public Health ───────────────────────────────
world = world.copy()
if "gdp_per_capita" in world.columns:
    world["disease_index"] = 100 * (1 - world["gdp_per_capita"].clip(upper=50000) / 50000)
else:
    rng = np.random.default_rng(42)
    world["disease_index"] = rng.uniform(10, 90, len(world))

world_proj = reproject_gdf(world, "robinson")

fig, ax = plt.subplots(figsize=(18,9))
fig.patch.set_facecolor("white")
ax.set_facecolor(STYLE["ocean_color"])
world_proj[world_proj["disease_index"].isna()].plot(ax=ax, color="#ddd", linewidth=0.2, edgecolor="#aaa")
world_proj.dropna(subset=["disease_index"]).plot(
    column="disease_index", ax=ax, scheme="Quantiles", k=5, cmap="YlOrRd",
    legend=True, legend_kwds={"title":"Disease Burden Index\n(100=highest)","loc":"lower left","fontsize":8},
    linewidth=0.2, edgecolor="#888", missing_kwds={"color":"#ddd"}
)
ax.set_axis_off()
add_map_annotations(ax, title="Scenario A — Public Health: Disease Burden Index by Country",
                    subtitle="Choropleth chosen: normalised country-level rate, suitable for policy-level comparison",
                    source="Derived from World Bank GDP 2020", projection_name="robinson", year=2020)
save_figure(fig, PATHS["fig_part7"] / "scenario_a_public_health.png")
plt.show()
print("Design decision: CHOROPLETH — disease burden is a country-level normalised rate, not a point phenomenon")

In [ ]:
# ── SCENARIO B: Urban Services ──────────────────────────────
top_ap = airports.nlargest(400, "total_routes").dropna(subset=["lat","lon"])
ap_gdf  = points_to_gdf(top_ap, "lat", "lon")
ap_proj = reproject_gdf(ap_gdf, "robinson")
coords  = np.array([(g.x, g.y) for g in ap_proj.geometry])
sizes   = scale_symbols(ap_proj["total_routes"], min_size=5, max_size=1200, method="area")

fig, ax = plt.subplots(figsize=(18,9))
fig.patch.set_facecolor("white")
ax.set_facecolor(STYLE["ocean_color"])
reproject_gdf(world, "robinson").plot(ax=ax, color=STYLE["land_color"],
    linewidth=0.3, edgecolor=STYLE["boundary_color"])

ax.scatter(coords[:,0], coords[:,1], s=sizes, c="#2c7bb6", alpha=0.65,
           edgecolors="white", linewidths=0.4, zorder=5)

for label, xy in [
    ("Central Africa\n(underserved)", (-1.5e6, -5e5)),
    ("Central Asia\n(limited access)", (7e6, 4.5e6)),
    ("Pacific Islands\n(isolated)", (1.5e7, -2e6)),
]:
    ax.annotate(label, xy=xy, fontsize=8.5, color="#cc0000", ha="center",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cc0000", alpha=0.8))

for ref in [20, 100, 250]:
    rs = scale_symbols(pd.Series([ref, top_ap["total_routes"].max()]), 5, 1200, "area")[0]
    ax.scatter([], [], s=rs, c="#2c7bb6", alpha=0.65, label=f"{ref} routes")
ax.legend(title="Routes (circle area)", loc="lower left", fontsize=8)
ax.set_axis_off()
add_map_annotations(ax, title="Scenario B — Urban Services: Global Airport Accessibility",
                    subtitle="Proportional symbols chosen: airports are POINT features — choropleth would misplace data across entire country",
                    source="OpenFlights.org", projection_name="robinson", year=2020)
save_figure(fig, PATHS["fig_part7"] / "scenario_b_urban_services.png")
plt.show()

In [ ]:
# ── SCENARIO C: Climate Risk (Bivariate) ───────────────────
world_wgs = world.to_crs("EPSG:4326") if world.crs.to_epsg() != 4326 else world.copy()
rbf = RBFInterpolator(np.column_stack([temps["lon"], temps["lat"]]),
                      temps["mean_temp_c"].values, kernel="thin_plate_spline", smoothing=5)
centroids = world_wgs.geometry.centroid
world_wgs["interp_temp"] = rbf(np.column_stack([centroids.x, centroids.y]))

pop_n  = (world_wgs["pop_density"].fillna(0).clip(lower=0))
temp_n = world_wgs["interp_temp"].fillna(world_wgs["interp_temp"].mean())
pop_n  = (pop_n  - pop_n.min())  / (pop_n.max()  - pop_n.min()  + 1e-9)
temp_n = (temp_n - temp_n.min()) / (temp_n.max() - temp_n.min() + 1e-9)

world_wgs["pop_cls"]   = pd.qcut(pop_n,  3, labels=[0,1,2]).astype(int)
world_wgs["temp_cls"]  = pd.qcut(temp_n, 3, labels=[0,1,2]).astype(int)
world_wgs["bivar_cls"] = world_wgs["pop_cls"] + world_wgs["temp_cls"] * 3

BCOLORS = {0:"#e8e8e8",1:"#ace4e4",2:"#5ac8c8",
           3:"#dfb0d6",4:"#a5add3",5:"#5698b9",
           6:"#be64ac",7:"#8c62aa",8:"#3b4994"}
world_wgs["bcolor"] = world_wgs["bivar_cls"].map(BCOLORS).fillna("#ddd")

world_biv = reproject_gdf(world_wgs, "robinson")
fig, ax = plt.subplots(figsize=(18,9))
fig.patch.set_facecolor("white")
ax.set_facecolor(STYLE["ocean_color"])
for color, grp in world_biv.groupby("bcolor"):
    grp.plot(ax=ax, color=color, linewidth=0.25, edgecolor=STYLE["boundary_color"])

# Legend matrix
lax = fig.add_axes([0.06, 0.1, 0.11, 0.11])
for r in range(3):
    for c in range(3):
        lax.add_patch(mpatches.Rectangle((c,r),1,1, fc=BCOLORS.get(c+r*3,"#eee"), ec="white", lw=0.8))
lax.set_xlim(0,3); lax.set_ylim(0,3)
lax.set_xticks([0.5,1.5,2.5]); lax.set_xticklabels(["Low","Med","High"],fontsize=7)
lax.set_yticks([0.5,1.5,2.5]); lax.set_yticklabels(["Low","Med","High"],fontsize=7)
lax.set_xlabel("Temp (hazard)",fontsize=7); lax.set_ylabel("Pop. Density (exposure)",fontsize=7)
lax.set_title("Risk Matrix",fontsize=7,pad=2); lax.tick_params(length=0)

ax.set_axis_off()
add_map_annotations(ax, title="Scenario C — Climate Risk: Bivariate Hazard × Exposure Map",
                    subtitle="Dark blue = HIGH temperature AND HIGH population density = highest climate risk",
                    source="OWID / Berkeley Earth 2020", projection_name="robinson", year=2020)
save_figure(fig, PATHS["fig_part7"] / "scenario_c_climate_risk.png")
plt.show()

## Design Decision Summary

| Scenario | Map Type Chosen | Why | Why Not Alternative |
|---|---|---|---|
| A — Public Health | Choropleth | Normalised country-level rate | Proportional symbol would show raw counts, misleading |
| B — Urban Services | Proportional Symbols | Airports are point features | Choropleth implies uniform country-wide access |
| C — Climate Risk | Bivariate Choropleth | Risk = hazard × exposure (two variables) | Single-variable map loses critical combined signal |